In [22]:
import os, sys

import pandas as pd
import numpy as np
parts=os.getcwd().split('/')
print(parts)
if (parts[-1]=='scripts') |(parts[-1]=='notebooks'):
    parts=parts[:-1]
srcpath='/'.join(parts)+'/results/'
dname='diabetes'
exp_type='random'
respath='%s/%s_%s_unlearning_0.csv'%(srcpath, dname, exp_type)
res_df=pd.read_csv(respath, sep='\t')
colnames=list(res_df.columns)
print(len(colnames))
exp_model_specs=[ 'iter','solver_type',  'prot_per_class']
model_perf_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'num_unlearned_samples', 
               'tr_M0_Bacc', 'tr_M1_Bacc', 'tr_M2_Bacc', 'tr_M0_AUC', 'tr_M1_AUC',  'tr_M2_AUC',
               'tr_M0_nAcc', 'tr_M1_nAcc','tr_retain_corr_M1', 'tr_imp_corr_M1',  'tr_lost_preds_M1',
               'tr_M2_nAcc', 'tr_imp_corr_M2',  'tr_retain_corr_M2','tr_lost_preds_M2' ,
               'te_M0_Bacc', 'te_M1_Bacc', 'te_M2_Bacc', 'te_M0_AUC', 'te_M1_AUC',  'te_M2_AUC',
               'te_M0_nAcc', 'te_M1_nAcc','te_retain_corr_M1', 'te_imp_corr_M1',  'te_lost_preds_M1',
               'te_M2_nAcc', 'te_imp_corr_M2',  'te_retain_corr_M2','te_lost_preds_M2' 
]

model_param_cols=['et_retrain', 'et_unlearn', 'n_retrain', 'num_unlearned_samples', 
               'prot_dev_01', 'prot_dev_02', 'prot_dev_12' ]
res_df.head(3)

['', 'mnt', 'nvme1n1p2', 'codes_datasets', 'toolboxes', 'unlearn-lvq', 'notebooks']
41


,solver_type,prot_per_class,num_unlearned_samples,iter,et_retrain,et_unlearn,prot_dev_01,prot_dev_02,prot_dev_12,n_retrain,...,te_M2_nAcc,te_retain_corr_M2,te_lost_preds_M2,te_imp_corr_M2,te_M0_Bacc,te_M1_Bacc,te_M2_Bacc,te_M0_AUC,te_M1_AUC,te_M2_AUC
0,lbfgs,1,9,0,0.061703,0.000244,0.000952,1.218222,1.218415,81403,...,11588.0,7454,4359,4134,0.5499,0.5501,0.5706,0.6372,0.6372,0.6045
1,lbfgs,1,9,1,0.081720,0.000289,0.000680,0.012226,0.012357,81403,...,11805.0,11783,30,22,0.5499,0.5499,0.5493,0.6372,0.6372,0.6372
2,lbfgs,1,9,2,0.088945,0.000304,0.000608,0.010271,0.010562,81403,...,11808.0,11788,25,20,0.5499,0.5500,0.5495,0.6372,0.6372,0.6372


In [65]:
ppc=1
#sel_col=['solver_type', 'prot_per_class', 'num_unlearned_samples', 'iter','n_retrain', 'tr_M0_nAcc',
#        'tr_M1_nAcc', 'tr_retain_corr_M1','tr_lost_preds_M1', 'tr_imp_corr_M1',
#        'tr_M2_nAcc', 'tr_retain_corr_M2','tr_lost_preds_M2', 'tr_imp_corr_M2', 
#        'te_M0_nAcc',
#        'te_M1_nAcc', 'te_retain_corr_M1','te_lost_preds_M1', 'te_imp_corr_M1',
#        'te_M2_nAcc', 'te_retain_corr_M2','te_lost_preds_M2', 'te_imp_corr_M2']
temp_df=res_df[exp_model_specs+model_perf_cols].loc[(res_df['solver_type']=='lbfgs')]
models, subsets=['M1', 'M2'], ['tr', 'te']
eval_metrics=[]
for subset in subsets:
    if subset== subsets[0]:
        res_df['speedupX']=res_df['et_retrain']/res_df['et_unlearn']
        eval_metrics.append('speedupX')
        res_df['%s_Acc_M0'%(subset)]=res_df['%s_M0_nAcc'%(subset)]/res_df['n_retrain']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    else:
        res_df['%s_Acc_M0'%(subset)]=res_df['%s_M0_nAcc'%(subset)]/res_df['n_test']  #'n_test'
        eval_metrics.append('%s_Acc_M0'%(subset))
    for model in models:
        if subset== subsets[0]:
            res_df['%s_Acc_%s'%(subset, model)]=res_df['%s_%s_nAcc'%(subset, model)]/res_df['n_retrain']
        else:
         #print(subset, res_df['te_%s_nAcc'%(model), ].iloc[:2])
            res_df['%s_Acc_%s'%(subset, model)]=res_df['%s_%s_nAcc'%(subset, model)]/res_df['n_test']
        eval_metrics.append('%s_Acc_%s'%(subset, model))
#temp_df['%s_Acc_%s'%(subset, model)]=temp['%s_%s_nAcc'%(subset, model)]/temp['n_retrain']
#temp_df['%s_Acc_%s'%(subset, model2)]=temp['%s_%s_nAcc'%(subset, model2)]/temp['n_retrain']
        res_df['%s_retrain_frac_%s'%(subset, model)]=res_df['%s_retain_corr_%s'%(subset, model)]/res_df['%s_M0_nAcc'%subset]*100
        eval_metrics.append('%s_retrain_frac_%s'%(subset, model))
        res_df['%s_loss_frac_%s'%(subset, model)]=res_df['%s_lost_preds_%s'%(subset, model)]/res_df['%s_M0_nAcc'%subset]*100
        eval_metrics.append('%s_loss_frac_%s'%(subset, model))
        res_df['%s_gain_frac_%s'%(subset, model)]=res_df['%s_imp_corr_%s'%(subset, model)]/res_df['%s_M0_nAcc'%subset]*100
        eval_metrics.append('%s_gain_frac_%s'%(subset, model))

#new_prf_metrics=itertools.chain()
sorted(np.setdiff1d(list(temp_df.columns), np.union1d(exp_model_specs, model_perf_cols)))
print(new_prf_metrics)

['te_gain_frac_M1', 'te_gain_frac_M2', 'te_loss_frac_M1', 'te_loss_frac_M2', 'te_retrain_frac_M1', 'te_retrain_frac_M2', 'tr_gain_frac_M1', 'tr_gain_frac_M2', 'tr_loss_frac_M1', 'tr_loss_frac_M2', 'tr_retrain_frac_M1', 'tr_retrain_frac_M2']


In [66]:
res_df.groupby(["solver_type","prot_per_class"])[["et_retrain","et_unlearn"]+eval_metrics].mean().reset_index()

,solver_type,prot_per_class,et_retrain,et_unlearn,speedupX,tr_Acc_M0,tr_Acc_M1,tr_retrain_frac_M1,tr_loss_frac_M1,tr_gain_frac_M1,...,tr_gain_frac_M2,te_Acc_M0,te_Acc_M1,te_retrain_frac_M1,te_loss_frac_M1,te_gain_frac_M1,te_Acc_M2,te_retrain_frac_M2,te_loss_frac_M2,te_gain_frac_M2
0,lbfgs,1,0.059434,0.000530,171.398141,0.580022,0.580341,99.849246,0.150754,0.205727,...,1.901944,0.580377,0.580757,99.847908,0.152092,0.217557,0.576111,97.405683,2.594317,1.859251
1,lbfgs,2,0.392492,0.000822,610.250841,0.589902,0.593676,97.745528,2.254472,2.894414,...,1.082381,0.591137,0.593695,97.603613,2.396387,2.829122,0.587339,98.392620,1.607380,0.964927
2,lbfgs,3,0.724624,0.000981,1069.403147,0.596788,0.593452,96.881329,3.118671,2.559841,...,4.555754,0.596296,0.593621,96.957787,3.042213,2.593722,0.596918,95.417319,4.582681,4.687045
3,sgd,1,0.881519,0.001765,611.669169,0.587737,0.589030,99.610284,0.389716,0.609854,...,4.846120,0.589123,0.590487,99.633336,0.366664,0.598226,0.561580,90.705251,9.294749,4.619576
4,sgd,2,1.033865,0.002948,398.244430,0.601995,0.604738,98.330474,1.669526,2.125236,...,5.931927,0.601209,0.603778,98.398845,1.601155,2.028547,0.581709,90.905995,9.094005,5.850563
5,sgd,3,1.168470,0.004521,283.521703,0.609659,0.608353,97.830015,2.169985,1.955684,...,5.609712,0.608038,0.607309,97.890810,2.109190,1.989334,0.595136,92.242512,7.757488,5.635639
6,wgd,1,0.029736,0.000756,60.999987,0.615423,0.615413,99.863328,0.136672,0.134954,...,7.789783,0.614769,0.614580,99.863076,0.136924,0.106289,0.598171,89.424332,10.575668,7.875809
7,wgd,2,0.031642,0.001002,41.791444,0.615869,0.616121,99.330472,0.669528,0.710405,...,9.226950,0.613786,0.614022,99.324422,0.675578,0.714000,0.593033,87.279010,12.720990,9.339897
8,wgd,3,0.034844,0.001251,38.302747,0.616550,0.616342,99.535488,0.464512,0.430711,...,13.028170,0.614228,0.614019,99.550739,0.449261,0.415134,0.582940,81.742121,18.257879,13.164027


In [67]:
res_df.groupby(["solver_type","prot_per_class"])[["et_retrain","et_unlearn"]+eval_metrics].std().reset_index()

,solver_type,prot_per_class,et_retrain,et_unlearn,speedupX,tr_Acc_M0,tr_Acc_M1,tr_retrain_frac_M1,tr_loss_frac_M1,tr_gain_frac_M1,...,tr_gain_frac_M2,te_Acc_M0,te_Acc_M1,te_retrain_frac_M1,te_loss_frac_M1,te_gain_frac_M1,te_Acc_M2,te_retrain_frac_M2,te_loss_frac_M2,te_gain_frac_M2
0,lbfgs,1,0.018493,0.000368,115.168522,0.000332,0.000721,0.146637,0.146637,0.261456,...,6.362499,0.0,0.000809,0.162460,0.162460,0.258549,0.005418,6.673599,6.673599,6.297557
1,lbfgs,2,0.294604,0.000434,613.909775,0.000311,0.003690,0.583579,0.583579,1.089600,...,1.289469,0.0,0.003438,0.649825,0.649825,1.090713,0.005005,1.675240,1.675240,1.189290
2,lbfgs,3,0.674141,0.000545,1312.489235,0.000266,0.006826,1.489805,1.489805,0.741154,...,3.758991,0.0,0.006399,1.544546,1.544546,0.769276,0.008828,2.999195,2.999195,3.765440
3,sgd,1,0.071916,0.000893,254.459651,0.000464,0.001306,0.405770,0.405770,0.657374,...,1.320532,0.0,0.001390,0.410087,0.410087,0.637367,0.016688,3.131320,3.131320,1.323180
4,sgd,2,0.070241,0.000977,158.858883,0.000382,0.002217,1.162779,1.162779,1.441408,...,1.549958,0.0,0.002011,1.161692,1.161692,1.398393,0.007734,2.474623,2.474623,1.577164
5,sgd,3,0.087600,0.001282,98.783367,0.000525,0.001817,0.848614,0.848614,0.881450,...,3.108589,0.0,0.002098,0.886586,0.886586,0.869792,0.006749,4.078157,4.078157,3.177972
6,wgd,1,0.004848,0.000643,35.220231,0.000394,0.000435,0.183129,0.183129,0.165561,...,7.704831,0.0,0.000192,0.162941,0.162941,0.156156,0.028051,12.189807,12.189807,7.718179
7,wgd,2,0.005670,0.000566,22.572514,0.000605,0.000703,0.185085,0.185085,0.212167,...,8.504974,0.0,0.000309,0.195229,0.195229,0.225407,0.031932,13.595217,13.595217,8.474639
8,wgd,3,0.005027,0.000702,22.372588,0.000447,0.000506,0.131291,0.131291,0.131268,...,6.611315,0.0,0.000407,0.147715,0.147715,0.123261,0.024446,10.615623,10.615623,6.683316
